<a href="https://colab.research.google.com/github/Paulo123213/MineriaDatos/blob/TomasOrtega/prueba2mineria.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#  Minería de Datos — TETR.IO: Predicción de Rango de Jugadores
### Metodología CRISP-DM
---
**Dataset:** TETRIO Tetra League Historical Data (Kaggle)  
**Objetivo:** Predecir el rango numérico de un jugador a partir de sus métricas de juego (APM, PPS, APP)


---
## 📌 FASE 1 — Comprensión del Negocio
### 1.1 Contexto del problema
TETR.IO es un juego de puzzles competitivo (tipo Tetris) con un sistema de ranking. El rango de un jugador refleja su habilidad relativa frente a otros en modalidad 1 vs 1.

### 1.2 Objetivo del negocio
Determinar qué métricas de desempeño en partida (APM, PPS, APP) son los mejores predictores del rango de un jugador, y construir un modelo capaz de estimarlo.

### 1.3 Criterio de éxito
- Correlación entre APM y rango > 0.70  
- Modelo de predicción con R² > 0.70

### 1.4 Glosario de Variables
| Variable | Descripción |
|---|---|
| `apm` | Ataques por Minuto |
| `pps` | Piezas por Segundo |
| `raw_app` | Ataques por Pieza (sin promediar) |
| `adjusted_app` | Ataques por Pieza (promediado entre partidas) |
| `tr` | Tetra Rating del jugador |
| `glicko` | Representación numérica del rating |
| `rd` | Rating Deviation |
| `tlrank` | Rango textual (d, c, b, a, s, ss, u, x) |
| `rank_num` | Rango mapeado a valor numérico (0–17) |


---
## 📊 FASE 2 — Comprensión de los Datos
### 2.1 Carga del dataset


In [ ]:
import numpy as np
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.ensemble import HistGradientBoostingRegressor
import kagglehub

# Descarga del dataset desde Kaggle
path = kagglehub.dataset_download("limguowei/tetrio-tetra-league-historical-data")
print("Archivos disponibles:", os.listdir(path))

df = pd.read_csv(os.path.join(path, 'tl-calculated-delta-table.csv'))
print(f"\nShape del dataset: {df.shape}")
df.head(10)


100%|██████████| 1.01G/1.01G [00:14<00:00, 77.2MB/s]

Extracting files...


### 2.2 Exploración inicial

In [ ]:
print("=== Información general ===")
print(df.info())
print("\n=== Valores nulos ===")
print(df.isna().sum())
print("\n=== Estadísticas descriptivas ===")
df.describe()


### 2.3 Distribución de la variable objetivo (tlrank)

In [ ]:
orden_rangos = ['z','d','d+','c-','c','c+','b-','b','b+','a-','a','a+','s-','s','s+','ss','u','x']

plt.figure(figsize=(12, 5))
df['tlrank'].value_counts().reindex(orden_rangos, fill_value=0).plot(kind='bar', color='steelblue')
plt.title('Distribución de Jugadores por Rango (tlrank)', fontsize=14, fontweight='bold')
plt.xlabel('Rango')
plt.ylabel('Cantidad de Jugadores')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


---
## 🔧 FASE 3 — Preparación de los Datos
### 3.1 Limpieza: eliminación de columnas irrelevantes


In [ ]:
df_clean = df.copy()

# Columnas irrelevantes o con información duplicada
df_clean = df_clean.drop(columns=['vs', 'raw_eff', 'adjusted_eff', 'verified'])

# Normalización del rango textual
df_clean['tlrank'] = df_clean['tlrank'].astype(str).str.strip().str.lower()

print("Columnas restantes:", df_clean.columns.tolist())
print(f"Shape: {df_clean.shape}")


### 3.2 Codificación ordinal de la variable objetivo (tlrank → rank_num)

In [ ]:
rank_mapping = {
    'x': 17, 'u': 16, 'ss': 15, 's+': 14, 's': 13, 's-': 12,
    'a+': 11, 'a': 10, 'a-': 9,  'b+': 8,  'b': 7,  'b-': 6,
    'c+': 5,  'c': 4,  'c-': 3,  'd+': 2,  'd': 1,  'z': 0
}

df_clean['rank_num'] = df_clean['tlrank'].map(rank_mapping)
print("Muestra del mapeo:")
df_clean[['tlrank', 'rank_num']].drop_duplicates().sort_values('rank_num')


### 3.3 Conversión de tipos y eliminación de NaN

In [ ]:
cols_analisis = ['rank_num', 'apm', 'pps', 'raw_app', 'adjusted_app']

for col in cols_analisis:
    df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce')

df_final = df_clean[cols_analisis].dropna()

print(f"Registros originales : {len(df_clean)}")
print(f"Registros tras dropna: {len(df_final)}")
print(f"Registros eliminados : {len(df_clean) - len(df_final)}")
df_final.head()


### 3.4 División Train / Test

In [ ]:
X_nuevo = df_final[['apm', 'pps', 'raw_app', 'adjusted_app']]
y_nuevo = df_final['rank_num']

X_train, X_test, y_train, y_test = train_test_split(
    X_nuevo, y_nuevo, test_size=0.2, random_state=42
)

# Visualización de la proporción
etiquetas  = ['Entrenamiento (80%)', 'Prueba (20%)']
cantidades = [X_train.shape[0], X_test.shape[0]]
colores    = ['#4CAF50', '#FFC107']

plt.figure(figsize=(5, 5))
plt.pie(cantidades, labels=etiquetas, colors=colores,
        autopct='%1.1f%%', startangle=140, shadow=True, explode=(0.05, 0))
plt.title('División Train / Test', fontsize=12, fontweight='bold')
plt.show()

print(f"Muestras de entrenamiento: {X_train.shape[0]}")
print(f"Muestras de prueba       : {X_test.shape[0]}")


---
## 🤖 FASE 4 — Modelado
### 4.1 Análisis de correlación (selección de variables)


In [ ]:
plt.figure(figsize=(10, 8))
matriz_corr = df_final.corr()
sns.heatmap(matriz_corr, annot=True, cmap='coolwarm', fmt=".2f", linewidths=0.5)
plt.title('Matriz de Correlación — Métricas de Juego TETR.IO', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("\nCorrelación con rank_num:")
print(matriz_corr['rank_num'].sort_values(ascending=False))


### 4.2 Entrenamiento del modelo — HistGradientBoostingRegressor
Se elige este modelo por su eficiencia con datasets grandes y su manejo nativo de valores faltantes.

In [ ]:
print("Entrenando HistGradientBoostingRegressor...")
modelo = HistGradientBoostingRegressor(max_iter=100, random_state=42)
modelo.fit(X_train, y_train)
print("Entrenamiento completado.")


---
## 📈 FASE 5 — Evaluación
### 5.1 Métricas de rendimiento


In [ ]:
y_pred_raw   = modelo.predict(X_test)
rango_minimo = y_train.min()
y_pred_final = np.clip(y_pred_raw, a_min=rango_minimo, a_max=None)

r2   = r2_score(y_test, y_pred_final)
mae  = mean_absolute_error(y_test, y_pred_final)
mse  = mean_squared_error(y_test, y_pred_final)
rmse = np.sqrt(mse)

df_metricas = pd.DataFrame([{
    'Modelo': 'HistGradientBoosting',
    'R²'    : round(r2,   4),
    'MAE'   : round(mae,  4),
    'MSE'   : round(mse,  4),
    'RMSE'  : round(rmse, 4)
}])
print("=== MÉTRICAS DEL MODELO ===")
print(df_metricas.to_string(index=False))


### 5.2 Análisis visual de outliers por rango

In [ ]:
plt.figure(figsize=(15, 8))
sns.set_theme(style="whitegrid")

sns.boxplot(x='tlrank', y='apm', data=df_clean,
            order=orden_rangos, palette="Spectral")
plt.title('Detección de Outliers: APM por Rango en TETR.IO', fontsize=16, fontweight='bold')
plt.xlabel('Rango', fontsize=12)
plt.ylabel('Ataques por Minuto (APM)', fontsize=12)
plt.tight_layout()
plt.show()


### 5.3 Relación APM vs Rango (dispersión y tendencia)

In [ ]:
df_grafico = df_final.sort_values('rank_num')

plt.figure(figsize=(12, 6))
sns.scatterplot(x='apm', y='rank_num', data=df_final,
                alpha=0.1, hue='rank_num', palette='viridis', legend=False)
sns.lineplot(data=df_grafico, x='rank_num', y='apm',
             color='darkblue', linewidth=3, label='Tendencia Promedio')
plt.title('Evolución del APM según el Rango del Jugador', fontsize=14, fontweight='bold')
plt.xlabel('Rango Numérico (0=D, 17=X)')
plt.ylabel('Ataques por Minuto (APM)')
plt.grid(True, linestyle='--', alpha=0.7)
plt.xticks(range(0, 18), rotation=45)
plt.legend()
plt.tight_layout()
plt.show()


### 5.4 Valores reales vs predichos

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

df_resultados = pd.DataFrame({
    'Rango Real'   : y_test.values,
    'Rango Predicho': y_pred_final
})

# Scatter real vs predicho
axes[0].scatter(y_test, y_pred_final, alpha=0.2, color='#1b4d3e')
axes[0].plot([y_test.min(), y_test.max()],
             [y_test.min(), y_test.max()], 'r--', lw=2, label='Predicción Perfecta')
axes[0].set_title('Valores Reales vs Predichos', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Rango Real')
axes[0].set_ylabel('Rango Predicho')
axes[0].legend()

# Tendencia promedio por rango
resumen = df_resultados.groupby('Rango Real')['Rango Predicho'].agg(['mean','std']).reset_index()
axes[1].plot(resumen['Rango Real'], resumen['mean'],
             marker='o', color='#1b4d3e', lw=3, label='Promedio Predicho')
axes[1].fill_between(resumen['Rango Real'],
                     resumen['mean'] - resumen['std'],
                     resumen['mean'] + resumen['std'],
                     color='#1b4d3e', alpha=0.15, label='±1 std')
axes[1].plot([y_test.min(), y_test.max()],
             [y_test.min(), y_test.max()], 'r--', lw=2, label='Predicción Perfecta')
axes[1].set_title('Tendencia: Rango Real vs Promedio Predicho', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Rango Real')
axes[1].set_ylabel('Rango Predicho Promedio')
axes[1].legend()
axes[1].grid(True, linestyle=':', alpha=0.6)

plt.tight_layout()
plt.show()


---
## 🚀 FASE 6 — Despliegue
### 6.1 Conclusiones del modelo

El análisis exploratorio y el modelo de regresión confirman una **relación positiva fuerte** entre las métricas de juego y el rango en TETR.IO:

| Hallazgo | Detalle |
|---|---|
| **Correlación APM–Rango** | ~0.82, la más alta del conjunto |
| **Modelo utilizado** | HistGradientBoostingRegressor (óptimo para datasets grandes) |
| **Variables predictoras** | `apm`, `pps`, `raw_app`, `adjusted_app` |
| **Corrección aplicada** | `np.clip` para evitar predicciones de rango negativo |

### 6.2 Próximos pasos sugeridos
1. **Incluir `glicko` y `tr`** como features adicionales para mejorar el R²  
2. **Probar clasificación** (en lugar de regresión) para predecir el rango exacto como categoría  
3. **Validación cruzada (k-fold)** para una estimación más robusta del error  
4. **Análisis temporal**: evaluar si el rendimiento promedio por rango ha cambiado con el tiempo
